# Eugen's SD2.1 Pick-a-Pic Experiment Runner (Colab)

Compares **Standard Sampling** vs **Z-Sampling** on the Pick-a-Pic benchmark using SD2.1.

Outputs are saved directly to Google Drive
Re-running any cell is safe — already generated images are skipped automatically due to the custom resume logic


## Cell 1 — Mount Google Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os
from getpass import getpass

REPO_DIR = "/content/generative-modelling"
BRANCH = "Run-SD2.1-experiment"

if not os.path.exists(REPO_DIR):
    try:
        print("Attempting to clone repository publicly...")
        subprocess.run(
            ["git", "clone", "https://github.com/viftode4/generative-modelling.git", REPO_DIR],
            check=True
        )
    except subprocess.CalledProcessError:
        print("\nClone failed. The repository is likely private.")
        token = getpass("Please enter your GitHub Personal Access Token (PAT): ")
        clone_url = f"https://{token}@github.com/viftode4/generative-modelling.git"
        subprocess.run(
            ["git", "clone", clone_url, REPO_DIR],
            check=True
        )

subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

print(f"Repo ready at {REPO_DIR} on branch {BRANCH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo ready at /content/generative-modelling on branch Run-SD2.1-experiment


In [ ]:
import os
print("Option A exists:", os.path.exists("/content/drive/MyDrive/eugen_pickapic_full"))
print("Option B exists:", os.path.exists("/content/drive/My Drive/eugen_pickapic_full"))

Option A exists: True
Option B exists: True


## Cell 2 — Install dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors tqdm pandas hpsv2
print("Dependencies installed.")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.8/316.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-iam 2.23.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
googleapis-common-protos 1.75.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
google-cloud-speech 2.3

## Cell 3 — Smoke test (2 prompts, 1 seed)

Verify the pipeline on T4 GPU before committing to the full run.

In [ ]:
import sys, os
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

SMOKE_OUT = "/content/drive/MyDrive/eugen_pickapic_smoke"

!PYTHONPATH=src python scripts/run_eugen_pickapic.py \
    --prompts experiments/prompts/pickapic_full.md \
    --seeds 42 \
    --limit 2 \
    --device cuda \
    --steps 50 \
    --cfg 7.5 \
    --out {SMOKE_OUT}

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading pipeline components...:  83% 5/6 [00:10<00:01,  1.62s/it]
Loading weights:   0% 0/372 [00:00<?, ?it/s]
Loading weights:   0% 1/372 [00:00<00:00, 1628.22it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   0% 1/372 [00:00<00:00, 1300.16it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   1% 2/372 [00:00<00:00, 964.10it/s, Materializing param=text_model.embeddings.token_embedding.weight]    
Loading weights:   1% 2/372 [00:00<00:00, 905.51it/s, Materializing param=text_model.embeddings.token_embedding.weight]
Loading weights:   1% 3/372 [00:01<03:15,  1.89it/s, Materializing param=text_model.embeddings.toke

## Cell 4 — Verify smoke test output

In [ ]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

manifest_path = Path(SMOKE_OUT) / "manifest.jsonl"
rows = [json.loads(l) for l in manifest_path.read_text().splitlines()]
print(f"{len(rows)} rows in manifest")

for row in rows:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(row["prompt"], fontsize=10)
    for ax, key, label in zip(axes,
        ["standard_path", "zigzag_path"],
        ["Standard", "Z-Sampling"]):
        ax.imshow(Image.open(row[key]))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Cell 5 — Full run (100 Pick-a-Pic prompts, 3 seeds)

Estimated runtime: ~2–3 hours on free Colab T4 GPU.

In [ ]:
# Run this cell to patch the diffusers output bug in Colab memory
path = "/content/generative-modelling/src/zigzag_project/zsampling.py"
with open(path, "r") as f:
    content = f.read()

patched = content.replace(
    "return StableDiffusionPipelineOutput(images=image)",
    "return StableDiffusionPipelineOutput(images=image, nsfw_content_detected=None)"
)

with open(path, "w") as f:
    f.write(patched)
print("zsampling.py successfully patched in Colab!")

zsampling.py successfully patched in Colab!


In [ ]:
%cd /content/generative-modelling

/content/generative-modelling


In [ ]:
FULL_OUT = "/content/drive/MyDrive/eugen_pickapic_full"

!PYTHONPATH=src python scripts/run_eugen_pickapic.py \
    --prompts experiments/prompts/pickapic_full.md \
    --seeds 42 123 2026 \
    --steps 50 \
    --cfg 7.5 \
    --inv-cfg 0 \
    --lambda-step 49 \
    --t-max 1 \
    --size 768 \
    --device cuda \
    --out {FULL_OUT}

Streaming output truncated to the last 5000 lines.
 90% 45/50 [00:46<00:05,  1.05s/it]
 92% 46/50 [00:48<00:04,  1.04s/it]
 94% 47/50 [00:49<00:03,  1.04s/it]
 96% 48/50 [00:50<00:02,  1.05s/it]
 98% 49/50 [00:51<00:01,  1.04s/it]
100% 50/50 [00:51<00:00,  1.03s/it]

  0% 0/50 [00:00<?, ?it/s]
  2% 1/50 [00:00<00:16,  2.95it/s]
  4% 2/50 [00:00<00:16,  2.91it/s]
  6% 3/50 [00:01<00:16,  2.89it/s]
  8% 4/50 [00:01<00:15,  2.89it/s]
 10% 5/50 [00:01<00:15,  2.88it/s]
 12% 6/50 [00:02<00:15,  2.88it/s]
 14% 7/50 [00:02<00:14,  2.88it/s]
 16% 8/50 [00:02<00:14,  2.87it/s]
 18% 9/50 [00:03<00:14,  2.86it/s]
 20% 10/50 [00:03<00:13,  2.87it/s]
 22% 11/50 [00:03<00:13,  2.87it/s]
 24% 12/50 [00:04<00:13,  2.87it/s]
 26% 13/50 [00:04<00:12,  2.86it/s]
 28% 14/50 [00:04<00:12,  2.87it/s]
 30% 15/50 [00:05<00:12,  2.87it/s]
 32% 16/50 [00:05<00:11,  2.86it/s]
 34% 17/50 [00:05<00:11,  2.86it/s]
 36% 18/50 [00:06<00:11,  2.86it/s]
 38% 19/50 [00:06<00:10,  2.85it/s]
 40% 20/50 [00:06<00:10,  2.85

## Cell 6 — Evaluate generated images with HPSv2

In [14]:
FULL_OUT = "/content/drive/MyDrive/eugen_pickapic_full"

!PYTHONPATH=src python scripts/score_hpsv2_pairs.py \
    --manifest {FULL_OUT}/manifest.jsonl \
    --out {FULL_OUT}/scores.jsonl

!PYTHONPATH=src python scripts/summarize_vlad_results.py \
    --scores {FULL_OUT}/scores.jsonl \
    --out {FULL_OUT}/summary.md

from IPython.display import Markdown, display
display(Markdown(filename=f"{FULL_OUT}/summary.md"))

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
hpsv2 OpenCLIP vocabulary file missing. Downloading to /usr/local/lib/python3.12/dist-packages/hpsv2/src/open_clip/bpe_simple_vocab_16e6.txt.gz...
Download successful.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
open_clip_pytorch_model.bin: 100% 3.94G/3.94G [00:43<00:00, 90.4MB/s]
HPS_v2.1_compressed.pt: 100% 1.97G/1.97G [01:04<00:00, 30.7MB/s]
/usr/local/lib/python3.12/dist-packages/hpsv2/img_score.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecat

# Vlad GenEval HPSv2 Summary

Overall pairs: 300
Overall Z win rate: 71.0%
Mean HPS delta (Z - standard): 0.0110

## By GenEval tag

| Tag | Pairs | Z wins | Standard wins | Ties | Z win rate | Mean delta |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| official | 300 | 213 | 87 | 0 | 71.0% | 0.0110 |

## Top Z-Sampling wins

### pickapic_008_official seed 123 (official)
Prompt: painting of female d&d bard performing magic ritual while holding skull
Delta: 0.0951
- Standard: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_008_official_seed123_standard.png`
- Z-Sampling: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_008_official_seed123_zigzag.png`

### pickapic_052_official seed 2026 (official)
Prompt: a mug made out of toothbrushes
Delta: 0.0818
- Standard: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_052_official_seed2026_standard.png`
- Z-Sampling: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_052_official_seed2026_zigzag.png`

### pickapic_064_official seed 2026 (official)
Prompt: A cat walks on an apple
Delta: 0.0783
- Standard: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_064_official_seed2026_standard.png`
- Z-Sampling: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_064_official_seed2026_zigzag.png`


## Top Z-Sampling losses

### pickapic_010_official seed 123 (official)
Prompt: A knitted forrest with foxes at sunset painted with pastel colors
Delta: -0.0405
- Standard: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_010_official_seed123_standard.png`
- Z-Sampling: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_010_official_seed123_zigzag.png`

### pickapic_007_official seed 42 (official)
Prompt: Luke Skywalker with a blue lightsaber fights robocop, moon high in the sky
Delta: -0.0313
- Standard: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_007_official_seed42_standard.png`
- Z-Sampling: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_007_official_seed42_zigzag.png`

### pickapic_029_official seed 2026 (official)
Prompt: A tree made of fire
Delta: -0.0269
- Standard: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_029_official_seed2026_standard.png`
- Z-Sampling: `/content/drive/MyDrive/eugen_pickapic_full/images/pickapic_029_official_seed2026_zigzag.png`



# Verification of checkpointing



In [ ]:
import os
from pathlib import Path

path = Path("/content/drive/MyDrive/eugen_pickapic_full/images")
if path.exists():
    images = list(path.glob("*.png"))
    std = [img for img in images if "_standard.png" in img.name]
    zig = [img for img in images if "_zigzag.png" in img.name]
    print(f"Total images saved in Drive: {len(images)}")
    print(f"Standard images: {len(std)}")
    print(f"Z-Sampling images: {len(zig)}")
    print(f"Completed prompt-seed pairs: {min(len(std), len(zig))} / 300")
else:
    print("Output folder not found on Google Drive. The run may not have started writing files yet.")

Output folder not found on Google Drive. The run may not have started writing files yet.
